# Final model — Focal gamma=1, TTA, weight decay 1e-4, learning rate 1e-3

5-fold training with the selected hyperparameters. Target embedding CV ~0.977.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import os
import sys
from pathlib import Path

def _load_project():
    my_drive = Path("/content/drive/MyDrive")
    init_candidates = [
        p for p in my_drive.rglob("colab_init.py")
        if (p.parent / "birdclef").is_dir()
    ]
    if init_candidates:
        init_path = min(init_candidates, key=lambda p: len(p.parts))
        spec = importlib.util.spec_from_file_location("colab_init", init_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return
    roots = [
        nb.parent.resolve()
        for nb in my_drive.rglob("02_download_and_extract_embeddings.ipynb")
        if (nb.parent / "birdclef").is_dir()
    ]
    if not roots:
        raise FileNotFoundError(
            "Unzip the full repository on Google Drive, then open a notebook from that folder."
        )
    root = min(roots, key=lambda p: len(p.parts))
    sys.path.insert(0, str(root))
    os.chdir(root)

_load_project()

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=True)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.3 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Unzipped TTA embeddings to /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [ ]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
import birdclef.paths as paths

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [ ]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()


In [ ]:
import birdclef.paths as paths
EPOCHS = 5
WINNING_GAMMA = 1
WINNING_WD = 1e-4
WINNING_LR = 1e-3
WINNING_CONTEXT = 1
INPUT_DIMENSION = 1536 * WINNING_CONTEXT
SAVE_DIR = paths.MODELS_DIR / "best_model"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(42)
fold_scores = []
all_fold_history = {}

for TRAIN_FOLD in range(5):
    best_auc = 0.0
    train_df = df_TTA[df_TTA["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df_TTA[df_TTA["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=True)
    valid_ds = BirdDataset(valid_df, is_tta=True)
    train_loader = DataLoader(
        train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True, prefetch_factor=4
    )
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    seed_everything(42)
    model = BirdMoE(input_dim=INPUT_DIMENSION, num_classes=NUM_CLASSES).to(device)
    criterion = FocalLoss(gamma=WINNING_GAMMA)
    optimizer = optim.Adam(model.parameters(), lr=WINNING_LR, weight_decay=WINNING_WD)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if torch.isnan(inputs).any():
                continue
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
        if current_auc > best_auc:
            best_auc = current_auc
            torch.save(model.state_dict(), save_path)
            print(f"New best model saved for fold {TRAIN_FOLD}")

    fold_scores.append(best_auc)
    all_fold_history[TRAIN_FOLD] = {
        "train_loss": fold_train_losses,
        "val_loss": fold_val_losses,
        "val_auc": fold_val_aucs,
    }

    onnx_save_path = SAVE_DIR / f"bird_moe_fold{TRAIN_FOLD}.onnx"
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, INPUT_DIMENSION).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            str(onnx_save_path),
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported fold {TRAIN_FOLD} ONNX to {onnx_save_path}")

print(f"FINAL CV SCORE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
curves_path = SAVE_DIR / "learning_curves_final.json"
with open(curves_path, "w") as f:
    json.dump(
        {
            "model": (
                f"Final best model (Focal gamma={WINNING_GAMMA}, WD={WINNING_WD}, "
                f"LR={WINNING_LR}, ctx={WINNING_CONTEXT}, TTA)"
            ),
            "metric_note": "Embedding-level CV — not comparable to Kaggle soundscape inference",
            "epochs": EPOCHS,
            "folds": {str(k): v for k, v in all_fold_history.items()},
        },
        f,
        indent=2,
    )
print(f"Learning curves JSON saved to {curves_path}")

100%|██████████| 7110/7110 [00:08<00:00, 825.94it/s] 


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/5 | Train Loss: 0.9024 | Val Loss: 1.3341 | Val AUC: 0.9754
New best model saved for fold 0
Epoch 02/5 | Train Loss: 0.6896 | Val Loss: 1.3392 | Val AUC: 0.9752
Epoch 03/5 | Train Loss: 0.6402 | Val Loss: 1.3529 | Val AUC: 0.9762
New best model saved for fold 0
Epoch 04/5 | Train Loss: 0.6133 | Val Loss: 1.3659 | Val AUC: 0.9775
New best model saved for fold 0
Epoch 05/5 | Train Loss: 0.5965 | Val Loss: 1.3329 | Val AUC: 0.9778
New best model saved for fold 0
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/learning_curves_fold0.png
Exported fold 0 ONNX to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/bird_moe_fold0.onnx


100%|██████████| 7110/7110 [00:09<00:00, 759.81it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/5 | Train Loss: 0.8971 | Val Loss: 1.3449 | Val AUC: 0.9754
New best model saved for fold 1
Epoch 02/5 | Train Loss: 0.6851 | Val Loss: 1.3380 | Val AUC: 0.9765
New best model saved for fold 1
Epoch 03/5 | Train Loss: 0.6262 | Val Loss: 1.3396 | Val AUC: 0.9766
New best model saved for fold 1
Epoch 04/5 | Train Loss: 0.5957 | Val Loss: 1.3472 | Val AUC: 0.9767
New best model saved for fold 1
Epoch 05/5 | Train Loss: 0.5778 | Val Loss: 1.3336 | Val AUC: 0.9761
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/learning_curves_fold1.png
Exported fold 1 ONNX to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/bird_moe_fold1.onnx


100%|██████████| 7110/7110 [00:10<00:00, 687.33it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/5 | Train Loss: 0.8991 | Val Loss: 1.3091 | Val AUC: 0.9760
New best model saved for fold 2
Epoch 02/5 | Train Loss: 0.6849 | Val Loss: 1.3248 | Val AUC: 0.9764
New best model saved for fold 2
Epoch 03/5 | Train Loss: 0.6387 | Val Loss: 1.3139 | Val AUC: 0.9756
Epoch 04/5 | Train Loss: 0.6113 | Val Loss: 1.3482 | Val AUC: 0.9730
Epoch 05/5 | Train Loss: 0.5911 | Val Loss: 1.3319 | Val AUC: 0.9750
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/learning_curves_fold2.png
Exported fold 2 ONNX to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/bird_moe_fold2.onnx


100%|██████████| 7110/7110 [00:09<00:00, 758.31it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/5 | Train Loss: 0.8956 | Val Loss: 1.3563 | Val AUC: 0.9773
New best model saved for fold 3
Epoch 02/5 | Train Loss: 0.6835 | Val Loss: 1.3659 | Val AUC: 0.9764
Epoch 03/5 | Train Loss: 0.6356 | Val Loss: 1.3589 | Val AUC: 0.9767
Epoch 04/5 | Train Loss: 0.6056 | Val Loss: 1.3803 | Val AUC: 0.9751
Epoch 05/5 | Train Loss: 0.5845 | Val Loss: 1.3521 | Val AUC: 0.9766
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/learning_curves_fold3.png
Exported fold 3 ONNX to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/bird_moe_fold3.onnx


100%|██████████| 7109/7109 [00:08<00:00, 862.09it/s] 


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/5 | Train Loss: 0.9029 | Val Loss: 1.3051 | Val AUC: 0.9774
New best model saved for fold 4
Epoch 02/5 | Train Loss: 0.6849 | Val Loss: 1.3174 | Val AUC: 0.9763
Epoch 03/5 | Train Loss: 0.6371 | Val Loss: 1.3335 | Val AUC: 0.9764
Epoch 04/5 | Train Loss: 0.6150 | Val Loss: 1.3395 | Val AUC: 0.9769
Epoch 05/5 | Train Loss: 0.5944 | Val Loss: 1.3316 | Val AUC: 0.9768
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/learning_curves_fold4.png
Exported fold 4 ONNX to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/bird_moe_fold4.onnx
FINAL CV SCORE: 0.9771 (+/- 0.0005)
Learning curves JSON saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/best_model/learning_curves_final.json
